# RFC Simulator and Paper-Reproduction Notebook

This notebook is designed for the current Recursive Fractal Cosmology repository architecture.

It supports two clearly separated paths:

1. **Current paper-reproduction path**: G, R, N, DownstreamPhysicalProjection, DimensionlessValidation, S, T, U, V, W, X, Y2, Z, QG.
2. **Legacy/development simulator path**: previous simulator modules A through Q, plus legacy G/N/R modules when present.

Expected files in the same folder:

- `SimulationConfigs.json`
- `Module_G_R_N_S_T_FrozenPacket.json`
- `ValidationScreens_U_V_W_X_Y2_Z_QG.json` optional but recommended

Global rule for the paper-reproduction path: modules consume the frozen packet. They do not retune it.

Important boundary: Module Y2 is exploratory candidate discovery. It must be frozen and retested as Y3 before independent validation claims.


In [ ]:
import json
import math
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown, HTML

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

BASE_DIR = Path.cwd()

EXPECTED_FILES = {
    "simulation_configs": "SimulationConfigs.json",
    "frozen_packet": "Module_G_R_N_S_T_FrozenPacket.json",
    "validation_screens": "ValidationScreens_U_V_W_X_Y2_Z_QG.json"
}

PAPER_REPRODUCTION_ORDER = [
    "G",
    "R",
    "N",
    "DownstreamPhysicalProjection",
    "DimensionlessValidation",
    "S",
    "T",
    "U",
    "V",
    "W",
    "X",
    "Y2",
    "Z",
    "QG"
]

LEGACY_DEVELOPMENT_ORDER = [
    "A",
    "B",
    "C",
    "D",
    "E",
    "F",
    "H",
    "I",
    "J",
    "K",
    "L",
    "M",
    "O",
    "P",
    "Q",
    "G_legacy",
    "N_legacy",
    "R_legacy"
]

LEGACY_MODULES = {
    "G_legacy", "N_legacy", "R_legacy"
}

CANONICAL_ORDER = PAPER_REPRODUCTION_ORDER

display(Markdown("## 1. Load repository JSON files"))
print("Working directory:", BASE_DIR)


In [ ]:
def load_json_file(filename, required=False):
    path = BASE_DIR / filename
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Required file not found: {filename}")
        return None, {"file": filename, "exists": False, "loaded": False, "error": None}
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data, {"file": filename, "exists": True, "loaded": True, "error": None}
    except Exception as exc:
        if required:
            raise
        return None, {"file": filename, "exists": True, "loaded": False, "error": str(exc)}


simulation_configs, simulation_status = load_json_file(EXPECTED_FILES["simulation_configs"], required=False)
frozen_packet, frozen_status = load_json_file(EXPECTED_FILES["frozen_packet"], required=False)
validation_file, validation_status = load_json_file(EXPECTED_FILES["validation_screens"], required=False)

load_status = pd.DataFrame([simulation_status, frozen_status, validation_status])
display(load_status)

if frozen_packet is None and simulation_configs is None:
    raise RuntimeError("No usable RFC JSON files were found. Put the JSON files in the same folder as this notebook.")

display(Markdown("Loaded files. Continuing with schema normalization."))


In [ ]:
def as_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


def normalize_name(name):
    return str(name).lower().replace("_", "").replace("-", "").replace(" ", "")


def unwrap_modules(config):
    if config is None:
        return []
    if isinstance(config, list):
        return config
    if isinstance(config, dict):
        for key in ["modules", "moduleConfigs", "simulationModules", "configs"]:
            if isinstance(config.get(key), list):
                return config[key]
    return []


simulation_modules = unwrap_modules(simulation_configs)


def module_name(module_obj):
    if not isinstance(module_obj, dict):
        return None
    for key in ["module", "id", "name", "moduleId", "moduleID"]:
        value = module_obj.get(key)
        if isinstance(value, str):
            return value
    return None


def available_config_module_names():
    names = []
    for obj in simulation_modules:
        name = module_name(obj)
        if name is not None:
            names.append(name)
    return names


def find_module_in_configs(name):
    target = normalize_name(name)
    for obj in simulation_modules:
        obj_name = module_name(obj)
        if obj_name and normalize_name(obj_name) == target:
            return obj
    return None


def find_key_contains(obj, tokens):
    if obj is None or not isinstance(obj, dict):
        return None
    tokens_low = [str(t).lower() for t in tokens]
    for key, value in obj.items():
        key_low = str(key).lower()
        if all(t in key_low for t in tokens_low):
            return value
    return None


def get_packet_module(name):
    if frozen_packet is None or not isinstance(frozen_packet, dict):
        return None

    target = normalize_name(name)
    direct_candidates = [
        name,
        str(name).lower(),
        f"module{name}",
        f"module_{name}",
        f"module{str(name).lower()}",
        f"module_{str(name).lower()}"
    ]
    for key in direct_candidates:
        if key in frozen_packet:
            return frozen_packet[key]

    for key, value in frozen_packet.items():
        key_clean = normalize_name(key)
        if key_clean == target or key_clean == normalize_name("module" + str(name)):
            return value
        if key_clean.startswith(normalize_name("module" + str(name))):
            return value

    if normalize_name(name) == "n":
        return find_key_contains(frozen_packet, ["module", "n"]) or find_key_contains(frozen_packet, ["dimensional"])
    if normalize_name(name) == "r":
        return find_key_contains(frozen_packet, ["module", "r"]) or find_key_contains(frozen_packet, ["closure", "audit"])
    if normalize_name(name) == "s":
        return find_key_contains(frozen_packet, ["module", "s"]) or find_key_contains(frozen_packet, ["anchor"])
    if normalize_name(name) == "t":
        return find_key_contains(frozen_packet, ["module", "t"]) or find_key_contains(frozen_packet, ["coupling"])

    return None


def raw_validation_block():
    if isinstance(validation_file, dict):
        if isinstance(validation_file.get("validationScreens"), dict):
            return validation_file.get("validationScreens")
        return validation_file
    if isinstance(frozen_packet, dict):
        if isinstance(frozen_packet.get("validationScreens"), dict):
            return frozen_packet.get("validationScreens")
    return {}


def infer_screen_id(key):
    clean = normalize_name(key)
    if clean in ["u", "v", "w", "x", "y2", "z", "qg"]:
        return clean.upper()
    ordered = ["Y2", "QG", "U", "V", "W", "X", "Z"]
    for candidate in ordered:
        c = candidate.lower()
        if clean.startswith(c):
            return candidate
        if clean.startswith("module" + c):
            return candidate
        if "module" + c in clean:
            return candidate
    return str(key)


def normalize_validation_screens():
    raw = raw_validation_block()
    normalized = {}
    if isinstance(raw, dict):
        for key, value in raw.items():
            normalized[infer_screen_id(key)] = value
    for name in ["U", "V", "W", "X", "Y2", "Z", "QG"]:
        if name not in normalized:
            cfg = find_module_in_configs(name)
            if cfg is not None:
                normalized[name] = cfg
    return normalized


validation_screens = normalize_validation_screens()

if isinstance(validation_file, dict) and isinstance(validation_file.get("validationScreens"), dict):
    VALIDATION_SOURCE = "standalone validation-screen file"
elif isinstance(frozen_packet, dict) and isinstance(frozen_packet.get("validationScreens"), dict):
    VALIDATION_SOURCE = "frozen-packet compatibility mirror"
else:
    VALIDATION_SOURCE = "SimulationConfigs fallback"

display(Markdown(f"**Using validation source:** {VALIDATION_SOURCE}"))


def get_downstream_projection():
    if isinstance(frozen_packet, dict):
        for key in ["downstreamPhysicalProjectionScreen", "downstreamPhysicalProjection", "combinedDownstreamPhysicalProjectionScreen"]:
            if key in frozen_packet:
                return frozen_packet[key]
        found = find_key_contains(frozen_packet, ["downstream", "projection"])
        if found is not None:
            return found
    return find_module_in_configs("DownstreamPhysicalProjection")


def get_dimensionless_validation():
    if isinstance(frozen_packet, dict):
        for key in ["dimensionlessValidationLayer", "dimensionlessValidation", "dimensionlessObservableValidationLayer"]:
            if key in frozen_packet:
                return frozen_packet[key]
        found = find_key_contains(frozen_packet, ["dimensionless", "validation"])
        if found is not None:
            return found
    return find_module_in_configs("DimensionlessValidation")


def get_module_data(name):
    if name == "DownstreamPhysicalProjection":
        return get_downstream_projection()
    if name == "DimensionlessValidation":
        return get_dimensionless_validation()
    if name in ["U", "V", "W", "X", "Y2", "Z", "QG"]:
        return validation_screens.get(name) or find_module_in_configs(name)

    # Current paper modules prefer frozen packet first.
    if name in ["G", "R", "N", "S", "T"]:
        return get_packet_module(name) or find_module_in_configs(name)

    # Legacy and A-Q modules come from SimulationConfigs.json.
    return find_module_in_configs(name) or get_packet_module(name)


def build_all_runnable_modules():
    config_names = available_config_module_names()
    ordered = []

    for name in PAPER_REPRODUCTION_ORDER:
        if name not in ordered:
            ordered.append(name)

    for name in LEGACY_DEVELOPMENT_ORDER:
        if any(normalize_name(name) == normalize_name(cn) for cn in config_names) and name not in ordered:
            ordered.append(name)

    for name in config_names:
        if name not in ordered:
            ordered.append(name)

    return ordered


ALL_RUNNABLE_MODULES = build_all_runnable_modules()

display(Markdown("## 2. Normalized package status"))

summary_rows = []
for name in PAPER_REPRODUCTION_ORDER:
    data = get_module_data(name)
    summary_rows.append({
        "path": "paper_reproduction",
        "component": name,
        "found": data is not None,
        "source": "frozen/validation/config auto-detected"
    })

for name in LEGACY_DEVELOPMENT_ORDER:
    data = get_module_data(name)
    if data is not None:
        summary_rows.append({
            "path": "legacy_development",
            "component": name,
            "found": True,
            "source": "SimulationConfigs.json"
        })

display(pd.DataFrame(summary_rows))
display(Markdown("Runnable module selector will include paper modules, legacy modules, and any extra modules found in `SimulationConfigs.json`."))


In [ ]:
def flatten_dict(obj, prefix=""):
    rows = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            next_prefix = f"{prefix}.{key}" if prefix else str(key)
            if isinstance(value, dict):
                rows.extend(flatten_dict(value, next_prefix))
            elif isinstance(value, list):
                if all(isinstance(x, dict) for x in value):
                    rows.append({"field": next_prefix, "value": f"list[{len(value)}] of records"})
                else:
                    rows.append({"field": next_prefix, "value": value})
            else:
                rows.append({"field": next_prefix, "value": value})
    else:
        rows.append({"field": prefix or "value", "value": obj})
    return rows


def find_first_record_list(obj):
    if isinstance(obj, list) and all(isinstance(x, dict) for x in obj):
        return obj
    if isinstance(obj, dict):
        priority_keys = [
            "results",
            "screenResults",
            "selectedResults",
            "scoredConstants",
            "constants",
            "rows",
            "quarks",
            "mixingAngles",
            "observerBranchingResults",
            "neuralEEGTargets"
        ]
        for key in priority_keys:
            if key in obj:
                found = find_first_record_list(obj[key])
                if found is not None:
                    return found
        for value in obj.values():
            found = find_first_record_list(value)
            if found is not None:
                return found
    return None


def extract_result_container(obj):
    if not isinstance(obj, dict):
        return obj
    for key in ["results", "screenResults", "selectedResults", "scoredConstants", "constants", "summary"]:
        if key in obj:
            return obj[key]
    return obj


def display_title(title):
    display(Markdown(f"## {title}"))


def display_note(text):
    display(Markdown(text))


def display_dataframe_safe(df, max_rows=200):
    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)
    if len(df) > max_rows:
        display(df.head(max_rows))
        display(Markdown(f"Showing first {max_rows} rows of {len(df)} total rows."))
    else:
        display(df)


def display_object(obj, title="Object"):
    display_title(title)
    if obj is None:
        display_note("**Not found.** Check that the relevant JSON file is present and populated.")
        return

    if isinstance(obj, dict) and isinstance(obj.get("summary"), dict):
        display_note("**Summary**")
        display_dataframe_safe(pd.DataFrame(flatten_dict(obj["summary"])))

    record_list = find_first_record_list(obj)
    if record_list is not None and len(record_list) > 0:
        display_dataframe_safe(pd.DataFrame(record_list))

    container = extract_result_container(obj)
    if isinstance(container, dict):
        rows = flatten_dict(container)
        display_dataframe_safe(pd.DataFrame(rows))
    elif isinstance(container, list):
        if all(isinstance(x, dict) for x in container):
            display_dataframe_safe(pd.DataFrame(container))
        else:
            display_dataframe_safe(pd.DataFrame({"value": container}))
    else:
        display(container)


def numeric_or_none(value):
    try:
        if isinstance(value, bool):
            return None
        return float(value)
    except Exception:
        return None


def deep_get_by_key(obj, target_keys):
    target_low = {str(k).lower() for k in as_list(target_keys)}
    if isinstance(obj, dict):
        for key, value in obj.items():
            if str(key).lower() in target_low:
                return value
        for value in obj.values():
            found = deep_get_by_key(value, target_keys)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = deep_get_by_key(value, target_keys)
            if found is not None:
                return found
    return None


def get_claim_boundary(obj):
    if isinstance(obj, dict):
        for key in ["claimBoundary", "boundary", "claimBoundarySummary", "interpretation", "status"]:
            if key in obj:
                return obj[key]
    return None


def print_boundary_if_present(obj):
    boundary = get_claim_boundary(obj)
    if boundary is not None:
        display_note("**Claim boundary / interpretation / status:**")
        display(boundary)


In [ ]:
def module_g_check():
    data = get_module_data("G")
    display_title("Module G: Deterministic Triadic Closure")
    if data is None:
        display_note("**Module G not found.**")
        return

    delta = numeric_or_none(deep_get_by_key(data, ["delta"]))
    cycle_length = numeric_or_none(deep_get_by_key(data, ["cycleLength", "cycle_length"]))
    phase_depth_k = numeric_or_none(deep_get_by_key(data, ["phaseDepthK", "phase_depth_k"]))
    alpha_packet = numeric_or_none(deep_get_by_key(data, ["alpha"]))
    nu_packet = numeric_or_none(deep_get_by_key(data, ["nu"]))
    epsilon_packet = numeric_or_none(deep_get_by_key(data, ["epsilon"]))
    empirical_targets = deep_get_by_key(data, ["empiricalTargetsUsed", "empirical_targets_used"])

    rows = []
    if delta is not None and cycle_length is not None and alpha_packet is not None:
        alpha_expected = math.log(delta) / cycle_length
        rows.append({"check": "alpha = log(delta) / cycleLength", "expected": alpha_expected, "packet": alpha_packet, "absError": abs(alpha_expected - alpha_packet)})
    if delta is not None and phase_depth_k is not None and nu_packet is not None:
        nu_expected = phase_depth_k * delta ** (-4)
        rows.append({"check": "nu = phaseDepthK * delta^(-4)", "expected": nu_expected, "packet": nu_packet, "absError": abs(nu_expected - nu_packet)})
    if alpha_packet is not None and nu_packet is not None and epsilon_packet is not None:
        epsilon_expected = alpha_packet * nu_packet
        rows.append({"check": "epsilon = alpha * nu", "expected": epsilon_expected, "packet": epsilon_packet, "absError": abs(epsilon_expected - epsilon_packet)})
    rows.append({"check": "empiricalTargetsUsed", "expected": False, "packet": empirical_targets, "absError": 0 if empirical_targets is False else "CHECK"})

    display_dataframe_safe(pd.DataFrame(rows))
    display_object(data, "Module G Stored Packet")


def module_r_check():
    data = get_module_data("R")
    display_title("Module R: Triad-Grouped Global Closure Audit")
    if data is None:
        display_note("**Module R not found.** This is a packaging problem if the paper reports Module R values.")
        return

    diagnostic_keys = [
        "rawRFLResidualScore",
        "sourceCoupledRFLResidualScore",
        "residualImprovement",
        "rawStandardizedResidualScore",
        "sourceCoupledRFLResidualScoreV2",
        "residualImprovementV2",
        "moduleRScoreV2",
        "cpResidual",
        "tailN18",
        "tailN40",
        "bestLagCorrelation",
        "bestLagCorrelationV1"
    ]
    rows = []
    for key in diagnostic_keys:
        value = deep_get_by_key(data, [key])
        if value is not None:
            rows.append({"field": key, "value": value})
    if rows:
        display_dataframe_safe(pd.DataFrame(rows))

    placeholder_flags = []
    for bad_key, bad_value in [("rawRFLResidualScore", 0.5), ("sourceCoupledRFLResidualScore", 0.5), ("residualImprovement", 0.0), ("tailN18", 0.0), ("tailN40", 0.0)]:
        val = numeric_or_none(deep_get_by_key(data, [bad_key]))
        if val is not None and abs(val - bad_value) < 1e-15:
            placeholder_flags.append(bad_key)
    if placeholder_flags:
        display_note("**WARNING:** possible placeholder Module R values detected: " + ", ".join(placeholder_flags))
    else:
        display_note("No obvious placeholder Module R values detected.")

    display_object(data, "Full Module R Object")


def generic_module_check(name, title=None):
    data = get_module_data(name)
    display_object(data, title or f"Module {name}")
    if name in LEGACY_MODULES or name in LEGACY_DEVELOPMENT_ORDER:
        display_note("**Legacy/development note:** This module is retained for continuity and exploration. It is not part of the current G/R/N/S/T paper-reproduction spine unless explicitly stated in the paper.")
    print_boundary_if_present(data)


def validation_screen_check(name):
    data = get_module_data(name)
    titles = {
        "U": "Module U: One-Anchor Constant Table Screen",
        "V": "Module V: Precision Cosmology Compressed-Parameter Screen",
        "W": "Module W: BBN Light-Abundance Proxy Screen",
        "X": "Module X: CP/EDM Bound Screen",
        "Y2": "Module Y2: Particle-Sector Refinement Screen",
        "Z": "Module Z: Observer, Branching, Neural, and EEG Harness",
        "QG": "Module QG: Finite Spin-Foam Transition-Amplitude Audit"
    }
    display_object(data, titles.get(name, f"Module {name}"))
    if name == "Y2":
        display_note("**Important:** Y2 is exploratory candidate discovery. It is not final independent validation until frozen and retested as Y3.")
    print_boundary_if_present(data)


def run_component(name):
    if name == "G":
        module_g_check()
    elif name == "R":
        module_r_check()
    elif name == "N":
        generic_module_check("N", "Module N V2: Dimensional Projection Bridge")
    elif name == "S":
        generic_module_check("S", "Module S: One-Anchor SI Bridge")
    elif name == "T":
        generic_module_check("T", "Module T: Dimensionless Coupling Map")
    elif name == "DownstreamPhysicalProjection":
        generic_module_check("DownstreamPhysicalProjection", "Downstream Physical-Projection Screen")
    elif name == "DimensionlessValidation":
        generic_module_check("DimensionlessValidation", "Dimensionless Observable Validation Layer")
    elif name in ["U", "V", "W", "X", "Y2", "Z", "QG"]:
        validation_screen_check(name)
    else:
        generic_module_check(name, f"Module {name}")


def run_all_paper():
    display(Markdown("# RFC Full Paper-Reproduction Pass"))
    for name in PAPER_REPRODUCTION_ORDER:
        run_component(name)
        display(HTML("<hr>"))


def run_legacy_development_modules():
    display(Markdown("# RFC Legacy / Development Simulator Pass"))
    ran_any = False
    for name in LEGACY_DEVELOPMENT_ORDER:
        data = get_module_data(name)
        if data is not None:
            ran_any = True
            run_component(name)
            display(HTML("<hr>"))
    if not ran_any:
        display_note("No legacy/development modules were found in SimulationConfigs.json.")


def run_all():
    run_all_paper()


display(Markdown("## 3. Module runners loaded"))
display(Markdown("Use `run_all_paper()` for the current paper path, `run_legacy_development_modules()` for A-Q and legacy G/N/R, or `run_component('A')` / `run_component('G_legacy')` manually."))


In [ ]:
def package_audit():
    display_title("Repository Package Audit")
    rows = []
    rows.append({"check": "SimulationConfigs.json loaded", "status": simulation_configs is not None})
    rows.append({"check": "Frozen packet JSON loaded", "status": frozen_packet is not None})
    rows.append({"check": "Standalone validation screens JSON loaded", "status": validation_file is not None})
    rows.append({"check": "Module G found", "status": get_module_data("G") is not None})
    rows.append({"check": "Module R found", "status": get_module_data("R") is not None})
    rows.append({"check": "Module N found", "status": get_module_data("N") is not None})
    rows.append({"check": "Module S found", "status": get_module_data("S") is not None})
    rows.append({"check": "Module T found", "status": get_module_data("T") is not None})
    rows.append({"check": "DownstreamPhysicalProjection found", "status": get_module_data("DownstreamPhysicalProjection") is not None})
    rows.append({"check": "DimensionlessValidation found", "status": get_module_data("DimensionlessValidation") is not None})
    for name in ["U", "V", "W", "X", "Y2", "Z", "QG"]:
        rows.append({"check": f"Validation screen {name} found", "status": get_module_data(name) is not None})

    legacy_found = []
    for name in LEGACY_DEVELOPMENT_ORDER:
        if get_module_data(name) is not None:
            legacy_found.append(name)
    rows.append({"check": "Legacy/development modules found", "status": len(legacy_found) > 0})

    df = pd.DataFrame(rows)
    display_dataframe_safe(df)

    missing = df[df["status"] == False]["check"].tolist()
    if missing:
        display_note("**Packaging warnings:**")
        for item in missing:
            display_note("- " + item)
        display_note("If a screen is reported in the paper, it should be present either in the standalone validation JSON or mirrored in the frozen packet JSON.")
    else:
        display_note("**Package audit passed:** all expected paper-reproduction components were found.")

    if validation_file is None:
        display_note("**Recommendation:** add `ValidationScreens_U_V_W_X_Y2_Z_QG.json` as a standalone file, even if the same screens are mirrored in the frozen packet.")

    if legacy_found:
        display_note("**Legacy/development modules detected:** " + ", ".join(legacy_found))
    else:
        display_note("No legacy/development modules detected. This is okay only if SimulationConfigs.json intentionally omits them.")


package_audit()


In [ ]:
if WIDGETS_AVAILABLE:
    display(Markdown("## 4. Interactive module selector"))
    selector = widgets.Dropdown(
        options=ALL_RUNNABLE_MODULES,
        value="G" if "G" in ALL_RUNNABLE_MODULES else ALL_RUNNABLE_MODULES[0],
        description="Component:",
        layout=widgets.Layout(width="620px")
    )
    run_button = widgets.Button(description="Run selected component", button_style="primary")
    run_paper_button = widgets.Button(description="Run paper reproduction", button_style="success")
    run_legacy_button = widgets.Button(description="Run legacy/development", button_style="warning")
    output = widgets.Output()

    def on_run_clicked(_):
        with output:
            output.clear_output()
            run_component(selector.value)

    def on_run_paper_clicked(_):
        with output:
            output.clear_output()
            run_all_paper()

    def on_run_legacy_clicked(_):
        with output:
            output.clear_output()
            run_legacy_development_modules()

    run_button.on_click(on_run_clicked)
    run_paper_button.on_click(on_run_paper_clicked)
    run_legacy_button.on_click(on_run_legacy_clicked)

    display(widgets.VBox([
        selector,
        widgets.HBox([run_button, run_paper_button, run_legacy_button])
    ]))
    display(output)
else:
    display(Markdown("## 4. Interactive widgets unavailable"))
    display(Markdown("Run modules manually with commands such as `run_component('G')`, `run_all_paper()`, or `run_legacy_development_modules()`."))


## Manual commands

Use these commands if the widget interface is not available:

```python
package_audit()

# Current paper-reproduction path
run_all_paper()
run_component('G')
run_component('R')
run_component('N')
run_component('DownstreamPhysicalProjection')
run_component('DimensionlessValidation')
run_component('S')
run_component('T')
run_component('U')
run_component('V')
run_component('W')
run_component('X')
run_component('Y2')
run_component('Z')
run_component('QG')

# Legacy/development simulator path
run_legacy_development_modules()
run_component('A')
run_component('B')
run_component('Q')
run_component('G_legacy')
run_component('N_legacy')
run_component('R_legacy')
```

Interpretation reminders:

- Module G is the current frozen deterministic packet source.
- Module R audits the frozen packet; it does not create the packet.
- Module N projects dimensional identities.
- Module S is a one-anchor SI bridge, not an independent prediction of the anchor.
- Module T maps the internal inverse-energy coupling into a dimensionless fine-structure-like value.
- Module U is a first-pass electromagnetic/atomic constant table screen.
- Modules V, W, X, Z, and QG are proxy/audit screens with explicit boundaries.
- Module Y2 is exploratory candidate discovery and must be frozen/retested as Y3 before independent validation claims.
- A-Q and legacy G/N/R modules are retained for development continuity and should not be confused with the current paper-reproduction spine.
